# Capítulo 5 — Agrupamento por Densidade: DBSCAN, OPTICS e HDBSCAN

Códigos do capítulo. Execute as células em ordem; cada bloco é independente dos demais.


## 5.3 DBSCAN no Python

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

X, _ = make_moons(n_samples=400, noise=0.07, random_state=0)
X = StandardScaler().fit_transform(X)

db = DBSCAN(eps=0.3, min_samples=5).fit(X)
rotulos = db.labels_                    # -1 significa ruído

n_grupos = len(set(rotulos)) - (1 if -1 in rotulos else 0)
print("Grupos encontrados:", n_grupos)
print("Pontos de ruído   :", (rotulos == -1).sum())

# Comparação com K-means
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(*X.T, c=km, s=15);      ax[0].set_title("K-means")
ax[1].scatter(*X.T, c=rotulos, s=15); ax[1].set_title("DBSCAN")
plt.show()

## 5.4.2 O gráfico de k-distâncias

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(make_moons(400, noise=0.07, random_state=0)[0])
k = 5
vizinhos = NearestNeighbors(n_neighbors=k).fit(X)
dist, _ = vizinhos.kneighbors(X)
dk = np.sort(dist[:, -1])               # distância ao k-ésimo vizinho

plt.plot(dk)
plt.xlabel("Pontos ordenados")
plt.ylabel(f"Distância ao {k}º vizinho")
plt.show()

## 5.6 OPTICS: a ordenação por alcançabilidade

In [ ]:
from sklearn.cluster import OPTICS
from sklearn.datasets import make_blobs
import numpy as np

# Três grupos com densidades bem diferentes
X1, _ = make_blobs(200, centers=[[0, 0]],  cluster_std=0.3, random_state=0)
X2, _ = make_blobs(200, centers=[[4, 4]],  cluster_std=1.0, random_state=1)
X3, _ = make_blobs(200, centers=[[-4, 4]], cluster_std=0.6, random_state=2)
X = np.vstack([X1, X2, X3])

op = OPTICS(min_samples=10, xi=0.05, min_cluster_size=0.05).fit(X)
print("Grupos:", len(set(op.labels_)) - (1 if -1 in op.labels_ else 0))
print("Ruído :", (op.labels_ == -1).sum())
# op.reachability_[op.ordering_] contém a curva do gráfico de alcançabilidade

## 5.7 HDBSCAN: densidade hierárquica

In [ ]:
from sklearn.cluster import HDBSCAN
from sklearn.datasets import make_blobs
import numpy as np

X1, _ = make_blobs(200, centers=[[0, 0]],  cluster_std=0.3, random_state=0)
X2, _ = make_blobs(200, centers=[[4, 4]],  cluster_std=1.0, random_state=1)
X = np.vstack([X1, X2])

h = HDBSCAN(min_cluster_size=20).fit(X)
print("Grupos:", len(set(h.labels_)) - (1 if -1 in h.labels_ else 0))
print("Ruído :", (h.labels_ == -1).sum())
print("Probabilidades (5 primeiros):", h.probabilities_[:5].round(2))

## 5.9 Estudo de caso: pontos de táxi em uma cidade

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

rng = np.random.default_rng(0)
# Coordenadas simuladas (graus): 3 pontos quentes + ruído espalhado
centros = np.array([[-23.55, -46.63], [-23.43, -46.47], [-23.62, -46.70]])
pts = np.vstack([c + rng.normal(0, 0.005, (150, 2)) for c in centros])
ruido = np.column_stack([rng.uniform(-23.7, -23.4, 60), rng.uniform(-46.8, -46.4, 60)])
coords = np.vstack([pts, ruido])

rad = np.radians(coords)
raio_terra_km = 6371.0
eps_km = 0.5                              # 500 metros
db = DBSCAN(eps=eps_km / raio_terra_km, min_samples=10,
            metric="haversine", algorithm="ball_tree").fit(rad)

print("Pontos quentes:", len(set(db.labels_)) - (1 if -1 in db.labels_ else 0))
print("Ruído:", (db.labels_ == -1).sum())

## Resumo do capítulo